# RetinaNet 모델

In [1]:
# 0. 환경 준비
!nvidia-smi

Thu Mar 19 04:47:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### git 연결

In [2]:
# 1. 내 포크 clone
!git clone https://github.com/wina0901/pill_detection_project.git

# 2. 프로젝트 폴더로 이동
%cd pill_detection_project

# 3. requirements 설치
!pip install --upgrade pip
!pip install --index-url https://download.pytorch.org/whl/cu118 torch==2.7.1+cu118 torchvision==0.22.1+cu118 torchaudio==2.7.1+cu118
!pip install -r requirements.txt --no-deps


Cloning into 'pill_detection_project'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 206 (delta 34), reused 35 (delta 15), pack-reused 135 (from 1)
Receiving objects: 100% (206/206), 499.20 KiB | 1.96 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/pill_detection_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 170.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 43.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 143.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 42.8 MB/s  0:00:07:00:01

### 구글 드라이브 연결

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import sys
import os
sys.path.append("/content/pill_detection_project")

from src.evaluation import (
    evaluate_all,
    init_history,
    update_history,
    save_history,
    load_history,
    plot_training_history,
    plot_compare_histories,
    convert_yolo_results,
    convert_torchvision_outputs,
)


# 데이터 가져오기

In [5]:
import os
from src.preprocessing.dataset import get_loaders

BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'  # Colab 기준

train_loader, val_loader, orig2model, num_classes = get_loaders(
    base_dir=BASE_DIR,
    batch_size=4,
    num_workers=2,
)

VAL_JSON_PATH = os.path.join(BASE_DIR, "val_letterbox.json")

✅ 고유 클래스 수  : 73종
✅ num_classes     : 74  ← 모델 정의 시 사용
✅ Train: 1790장 / 6134개
✅ Val  : 139장 / 431개


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


DEVICE: cuda


In [ ]:
############################################################
# 11. RetinaNet 모델 정의 + 학습
############################################################

from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.models.detection.retinanet import RetinaNetClassificationHead


def build_retinanet_model(num_classes):
    model = retinanet_resnet50_fpn(weights="DEFAULT")
    cls_head = model.head.classification_head

    # torchvision 버전에 따라 conv[0]이 Conv2dNormActivation일 수 있으므로
    # 내부 첫 Conv2d를 찾아 채널 수를 읽는다.
    first_conv = next(
        module for module in cls_head.conv.modules() if isinstance(module, nn.Conv2d)
    )
    in_channels = first_conv.in_channels
    num_anchors = cls_head.num_anchors

    model.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
    )
    return model


# 사전학습된 RetinaNet 모델 로드
model = build_retinanet_model(num_classes)
model.to(DEVICE)

# 옵티마이저 / 스케줄러 설정
retina_optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=1e-4,
)
retina_scheduler = optim.lr_scheduler.StepLR(retina_optimizer, step_size=3, gamma=0.1)


############################################################
# 12. RetinaNet 학습 루프
############################################################

num_epochs = 5

for epoch in range(1, num_epochs+1):
    ##########################
    # 1) Train
    ##########################
    model.train()
    train_loss_sum = 0.0

    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        retina_optimizer.zero_grad()
        losses.backward()
        retina_optimizer.step()

        train_loss_sum += losses.item()

    epoch_train_loss = train_loss_sum / max(1, len(train_loader))

    ##########################
    # 2) Validation (loss만 측정)
    ##########################
    model.train()  # detection 모델은 train() 상태에서 loss를 반환
    val_loss_sum = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss_sum += losses.item()

    epoch_val_loss = val_loss_sum / max(1, len(val_loader))

    print(
        f"[RetinaNet Epoch {epoch+1}/{num_epochs}] "
        f"train_loss: {epoch_train_loss:.4f} | "
        f"val_loss: {epoch_val_loss:.4f}"
    )

    retina_scheduler.step()

# 학습된 모델 저장
torch.save(model.state_dict(), "retinanet_oral_drug.pth")
print("RetinaNet 모델 저장 완료")



Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_coco-eeacb38b.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_coco-eeacb38b.pth


100%|██████████| 130M/130M [00:00<00:00, 190MB/s]  


[RetinaNet Epoch 2/5] train_loss: 0.6601 | val_loss: 0.4845
[RetinaNet Epoch 3/5] train_loss: 0.3949 | val_loss: 0.3373


In [ ]:
import json

with open(VAL_JSON_PATH, "r", encoding="utf-8") as f:
    val_coco = json.load(f)

val_image_ids = [img["id"] for img in val_coco["images"]]
print("val image count:", len(val_image_ids))
print("val_loader image count:", len(val_loader.dataset))


In [ ]:
history = init_history()

for epoch in range(1, num_epochs + 1):
    # 1. train
    train_loss = epoch_train_loss

    # 2. val
    val_loss = epoch_val_loss

    # 3. 검증셋 예측
    all_outputs = []
    all_image_ids = []

    model.eval()
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            outputs = model(images)

            batch_image_ids = [t["image_id"].item() for t in targets]

            all_outputs.extend(outputs)
            all_image_ids.extend(batch_image_ids)

    val_predictions = convert_torchvision_outputs(all_outputs, all_image_ids)

    # 4. 평가
    metrics = evaluate_all(
        gt_json_path=VAL_JSON_PATH,
        predictions=val_predictions,
        conf_threshold=0.25,
        pr_iou_threshold=0.5,
        temp_json_path=f"retinanet_temp_eval_epoch_{epoch}.json"
    )

    # 5. 기록
    update_history(
        history,
        epoch=epoch,
        train_loss=train_loss,
        val_loss=val_loss,
        metrics=metrics
    )

save_history(history, "history_retinanet.json")
plot_training_history(history, title_prefix="RetinaNet")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...


AssertionError: Results do not correspond to current coco set

In [ ]:
import json
import random
import textwrap
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

with open(VAL_JSON_PATH, "r", encoding="utf-8") as f:
    val_coco = json.load(f)

cat_id_to_name = {c["id"]: c["name"] for c in val_coco["categories"]}
model2orig = {v: k for k, v in orig2model.items()}

NUM_SAMPLES = 6
SCORE_THRESH = 0.3
N_COLS = 2
N_ROWS = 3

def wrap_label(prefix, name, width=18, score=None):
    wrapped_name = "\n".join(textwrap.wrap(str(name), width=width))
    if score is None:
        return f"{prefix}:\n{wrapped_name}"
    return f"{prefix}:\n{wrapped_name}\n({score:.2f})"

sample_indices = random.sample(
    range(len(val_loader.dataset)),
    k=min(NUM_SAMPLES, len(val_loader.dataset))
)

model.eval()

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(12, 16))
axes = axes.flatten()

with torch.no_grad():
    for ax, idx in zip(axes, sample_indices):
        image, target = val_loader.dataset[idx]
        output = model([image.to(DEVICE)])[0]

        img_np = image.permute(1, 2, 0).cpu().numpy()
        ax.imshow(img_np)
        ax.axis("off")

        gt_boxes = target["boxes"].cpu().numpy()
        gt_labels = target["labels"].cpu().numpy()

        for box, label in zip(gt_boxes, gt_labels):
            x1, y1, x2, y2 = box
            orig_cat = model2orig.get(int(label), int(label))
            pill_name = cat_id_to_name.get(orig_cat, str(orig_cat))

            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor="lime", facecolor="none"
            )
            ax.add_patch(rect)

            ax.text(
                x1, max(y1 - 5, 5),
                wrap_label("GT", pill_name, width=18),
                color="lime",
                fontsize=8,
                bbox=dict(facecolor="black", alpha=0.6, pad=2)
            )

        pred_boxes = output["boxes"].detach().cpu().numpy()
        pred_scores = output["scores"].detach().cpu().numpy()
        pred_labels = output["labels"].detach().cpu().numpy()

        for box, score, label in zip(pred_boxes, pred_scores, pred_labels):
            if score < SCORE_THRESH:
                continue

            x1, y1, x2, y2 = box
            orig_cat = model2orig.get(int(label), int(label))
            pill_name = cat_id_to_name.get(orig_cat, str(orig_cat))

            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor="red", facecolor="none"
            )
            ax.add_patch(rect)

            ax.text(
                x1, min(y2 + 12, img_np.shape[0] - 5),
                wrap_label("Pred", pill_name, width=18, score=score),
                color="red",
                fontsize=8,
                bbox=dict(facecolor="white", alpha=0.8, pad=2)
            )

        ax.set_title(f"Val Sample idx={idx} | green=GT | red=Prediction")

for ax in axes[len(sample_indices):]:
    ax.axis("off")

plt.tight_layout()
plt.show()
